<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/PlantVillage_ResNet50_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================
# 1. Kaggle Setup and Dataset Download
# ===============================
from google.colab import files
files.upload()  # Upload kaggle.json first

!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download PlantVillage dataset
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip -d PlantVillage

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
 94% 620M/658M [00:03<00:00, 201MB/s]
100% 658M/658M [00:04<00:00, 160MB/s]


In [ ]:
# ===============================
# 2. Imports
# ===============================
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing import image_dataset_from_directory

# Parameters
img_size = (224, 224)  # ResNet50 default input size
batch_size = 32

In [ ]:
# ===============================
# 3. Dataset Creation
# ===============================
dataset = image_dataset_from_directory(
    "/content/PlantVillage/PlantVillage",
    image_size=img_size,
    batch_size=batch_size,
    validation_split=0.2,  # 80% train, 20% val
    subset="training",
    seed=123
)
val_dataset = image_dataset_from_directory(
    "/content/PlantVillage/PlantVillage",
    image_size=img_size,
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    seed=123
)

# Split val -> val + test
val_batches = tf.data.experimental.cardinality(val_dataset)
test_dataset = val_dataset.take(val_batches // 2)
val_dataset = val_dataset.skip(val_batches // 2)

class_names = dataset.class_names
num_classes = len(class_names)
print("Classes:", class_names)
print("Num Classes:", num_classes)


Found 20638 files belonging to 15 classes.
Using 16511 files for training.
Found 20638 files belonging to 15 classes.
Using 4127 files for validation.
Classes: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']
Num Classes: 15


In [ ]:
# ===============================
# 4. Data Augmentation
# ===============================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

train_ds = dataset.map(lambda x, y: (data_augmentation(x), y))

# Prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_dataset = val_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [ ]:
# ===============================
# 5. Build ResNet50 Model
# ===============================
base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # freeze base initially

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

# One-hot encoding for labels
def one_hot_map(images, labels):
    return images, tf.one_hot(labels, depth=num_classes)

train_ds_onehot = train_ds.map(one_hot_map)
val_ds_onehot = val_dataset.map(one_hot_map)
test_ds_onehot = test_dataset.map(one_hot_map)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# ===============================
# 6. Initial Training
# ===============================
history = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=10
)

Epoch 1/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 266s 487ms/step - accuracy: 0.5447 - loss: 2.1723 - val_accuracy: 0.8456 - val_loss: 1.2674
Epoch 2/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 231s 444ms/step - accuracy: 0.7757 - loss: 1.4240 - val_accuracy: 0.8971 - val_loss: 1.1292
Epoch 3/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 260s 439ms/step - accuracy: 0.8097 - loss: 1.3206 - val_accuracy: 0.8725 - val_loss: 1.1410
Epoch 4/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 262s 439ms/step - accuracy: 0.8195 - loss: 1.2770 - val_accuracy: 0.8889 - val_loss: 1.0993
Epoch 5/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 232s 450ms/step - accuracy: 0.8181 - loss: 1.2684 - val_accuracy: 0.8841 - val_loss: 1.1029
Epoch 6/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 266s 457ms/step - accuracy: 0.8180 - loss: 1.2553 - val_accuracy: 0.8879 - val_loss: 1.0856
Epoch 7/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 229s 444ms/step - accuracy: 0.8252 - loss: 1.2488 - val_accuracy: 0.8903 - val_loss: 1.0799
Epoch 8/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 232s 448ms/step - accuracy: 0.8288 -

In [ ]:
# ===============================
# 7. Fine-Tuning
# ===============================
base_model.trainable = True
fine_tune_at = 100  # freeze first 100 layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=20,
    initial_epoch=10
)

Epoch 11/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 305s 529ms/step - accuracy: 0.7110 - loss: 1.5200 - val_accuracy: 0.9283 - val_loss: 1.0289
Epoch 12/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 256s 495ms/step - accuracy: 0.8742 - loss: 1.1478 - val_accuracy: 0.9447 - val_loss: 0.9741
Epoch 13/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 259s 490ms/step - accuracy: 0.9024 - loss: 1.0892 - val_accuracy: 0.9601 - val_loss: 0.9380
Epoch 14/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 257s 499ms/step - accuracy: 0.9121 - loss: 1.0572 - val_accuracy: 0.9644 - val_loss: 0.9177
Epoch 15/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 255s 494ms/step - accuracy: 0.9290 - loss: 1.0247 - val_accuracy: 0.9673 - val_loss: 0.9015
Epoch 16/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 264s 499ms/step - accuracy: 0.9408 - loss: 1.0036 - val_accuracy: 0.9760 - val_loss: 0.8856
Epoch 17/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 258s 500ms/step - accuracy: 0.9500 - loss: 0.9758 - val_accuracy: 0.9745 - val_loss: 0.8785
Epoch 18/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 255s 494ms/step - accuracy: 

In [ ]:
# ===============================
# 8. Evaluation with Temperature Scaling
# ===============================
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize

TEMPERATURE = 2.0

def predict_with_temperature(model, dataset, T=1.0):
    all_probs, all_labels = [], []
    for images, labels in dataset:
        logits = model(images, training=False)
        scaled_logits = logits / T
        probs = tf.nn.softmax(scaled_logits).numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_probs)

y_true, y_probs = predict_with_temperature(model, test_ds_onehot, T=TEMPERATURE)
y_pred = np.argmax(y_probs, axis=1)

# Convert one-hot labels to integers
y_true_int = np.argmax(y_true, axis=1)

y_true_bin = label_binarize(y_true_int, classes=range(num_classes))

accuracy = accuracy_score(y_true_int, y_pred)
precision = precision_score(y_true_int, y_pred, average="weighted")
recall = recall_score(y_true_int, y_pred, average="weighted")
f1 = f1_score(y_true_int, y_pred, average="weighted")
kappa = cohen_kappa_score(y_true_int, y_pred)
auc = roc_auc_score(y_true_bin, y_probs, average="macro", multi_class="ovr")

print("\n Evaluation Metrics After Calibration:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Kappa    : {kappa:.4f}")


 Evaluation Metrics After Calibration:
Accuracy : 0.9751
Precision: 0.9761
Recall   : 0.9751
F1 Score : 0.9750
AUC      : 0.9997
Kappa    : 0.9727
